# DIV2K Dataset Processing: BPP-based Image Selection

This notebook processes the DIV2K dataset to:
1. Compute BPP (Bits Per Pixel) for all images (fast pass, no SR storage)
2. Sort images by BPP
3. Select 10 diverse images based on BPP distribution
4. Apply super-resolution only to selected 10 images (efficient approach)

In [3]:
# Download DIV2K dataset
import os
import urllib.request
import zipfile
from pathlib import Path
import subprocess
import sys

# Install tqdm if not available (needed for progress bars)
try:
    from tqdm import tqdm
except ImportError:
    print("Installing tqdm for progress bars...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm", "-q"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    from tqdm import tqdm

# Create dataset directory
dataset_dir = Path("./DIV2K")
dataset_dir.mkdir(exist_ok=True)

# URL for downloading DIV2K (train HR images)
div2k_url = "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip"
zip_path = dataset_dir / "DIV2K_train_HR.zip"
images_dir = dataset_dir / "DIV2K_train_HR"

# Check if dataset already exists
if not images_dir.exists() or len(list(images_dir.glob("*.png"))) < 800:
    # Check if zip file exists and is valid
    zip_valid = False
    if zip_path.exists():
        try:
            # Test if file is a valid zip file
            with zipfile.ZipFile(zip_path, 'r') as test_zip:
                test_zip.testzip()
            zip_valid = True
            file_size_gb = zip_path.stat().st_size / (1024**3)
            print(f"Found existing zip file: {file_size_gb:.2f} GB (valid)")
        except (zipfile.BadZipFile, zipfile.LargeZipFile) as e:
            print(f"Existing file is not a valid zip file: {e}")
            print("Removing invalid file and re-downloading...")
            zip_path.unlink()
            zip_valid = False
    
    if not zip_valid:
        print("Downloading DIV2K dataset (~2GB)...")
        
        # Download with progress bar
        class DownloadProgressBar(tqdm):
            def update_to(self, b=1, bsize=1, tsize=None):
                if tsize is not None:
                    self.total = tsize
                self.update(b * bsize - self.n)
        
        try:
            with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc="Downloading") as t:
                urllib.request.urlretrieve(div2k_url, zip_path, reporthook=t.update_to)
            print("Download complete!")
            
            # Verify downloaded file
            print("Verifying downloaded file...")
            with zipfile.ZipFile(zip_path, 'r') as test_zip:
                test_zip.testzip()
            print("File verification successful!")
        except Exception as e:
            print(f"Error during download or verification: {e}")
            if zip_path.exists():
                zip_path.unlink()
            raise
    
    print("Extracting archive...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Get list of files to extract
            file_list = zip_ref.namelist()
            with tqdm(total=len(file_list), unit='files', desc="Extracting") as pbar:
                for file in file_list:
                    zip_ref.extract(file, dataset_dir)
                    pbar.update(1)
        print("Extraction complete!")
    except zipfile.BadZipFile as e:
        print(f"Error: Zip file is corrupted. Please delete {zip_path} and re-run this cell.")
        raise
else:
    print(f"Dataset already exists in {images_dir}")

print(f"Found images: {len(list(images_dir.glob('*.png')))}")

Existing file is not a valid zip file: File is not a zip file
Removing invalid file and re-downloading...


Downloading: 3.53GB [02:22, 24.7MB/s]                               


Download complete!
Verifying downloaded file...
File verification successful!
Extracting archive...


Extracting: 100%|██████████| 801/801 [00:12<00:00, 61.97files/s]

Extraction complete!
Found images: 800


In [4]:
# Install required packages if not available
import subprocess
import sys

# Install tqdm if not available
try:
    import tqdm
    print(f"tqdm already installed: {tqdm.__version__}")
except ImportError:
    print("Installing tqdm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"], 
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import tqdm
    print("tqdm installed!")

# Install compressai if not available
try:
    import compressai
    print(f"compressai already installed: {compressai.__version__}")
except ImportError:
    print("Installing compressai...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "compressai"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import compressai
    print("compressai installed!")

tqdm already installed: 4.67.1
Installing compressai...


/Users/tienti/miniconda3/envs/myenvpython311/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


compressai installed!


In [5]:
import torch
import math
import numpy as np
from PIL import Image
from torchvision import transforms
import glob
import copy
from pathlib import Path
from tqdm import tqdm

# Force CPU usage (for MacBook)
device = torch.device('cpu')
print(f"Using device: {device}")

# Load compression model
from compressai.zoo import models
model_name = "cheng2020-attn"
quality = 6
model_class = models[model_name]
model_compression = model_class(quality=quality, pretrained=True).to(device)
model_compression.eval()
for param in model_compression.parameters():
    param.requires_grad = False
print("Compression model loaded")

# Load Super-Resolution model (Swin2SR)
from transformers import AutoImageProcessor, Swin2SRForImageSuperResolution
processor = AutoImageProcessor.from_pretrained("caidas/swin2SR-classical-sr-x2-64", do_pad=False, pad_size=0)
model_sr = Swin2SRForImageSuperResolution.from_pretrained("caidas/swin2SR-classical-sr-x2-64")
model_sr = model_sr.to(device)
model_sr.eval()
print("Super-Resolution model loaded")

Using device: cpu
Downloading: "https://compressai.s3.amazonaws.com/models/v1/cheng2020_attn-mse-6-730501f2.pth.tar" to /Users/tienti/.cache/torch/hub/checkpoints/cheng2020_attn-mse-6-730501f2.pth.tar


100%|██████████| 121M/121M [00:26<00:00, 4.72MB/s]  


Compression model loaded


preprocessor_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`self.pad_size` attribute is deprecated and will be removed in v5. Use `self.size_divisor` instead
`self.pad_size` attribute is deprecated and will be removed in v5. Use `self.size_divisor` instead
`self.pad_size` attribute is deprecated and will be removed in v5. Use `self.size_divisor` instead


config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/48.5M [00:00<?, ?B/s]

Super-Resolution model loaded


In [6]:
# Helper functions to replace custom library functions
def bpp_loss(compression_output, num_pixels):
    """Compute BPP from compression model output"""
    y_likelihood = compression_output['likelihoods']['y']
    z_likelihood = compression_output['likelihoods']['z']
    bpp = (torch.log(y_likelihood).sum() + torch.log(z_likelihood).sum()) / (-math.log(2) * num_pixels)
    return bpp

def ssim(img1, img2):
    """Compute SSIM metric (simplified version)"""
    # Use simple SSIM from scikit-image
    from skimage.metrics import structural_similarity as ssim_skimage
    img1_np = img1.squeeze().cpu().detach().numpy().transpose(1, 2, 0)
    img2_np = img2.squeeze().cpu().detach().numpy().transpose(1, 2, 0)
    img1_np = np.clip(img1_np, 0, 1)
    img2_np = np.clip(img2_np, 0, 1)
    if img1_np.shape[2] == 3:
        ssim_val = ssim_skimage(img1_np, img2_np, channel_axis=2, data_range=1.0)
    else:
        ssim_val = ssim_skimage(img1_np, img2_np, data_range=1.0)
    return torch.tensor(ssim_val)

print("Helper functions ready")

Helper functions ready


In [11]:
# Parameters and helper functions
dataset_path = "./DIV2K/DIV2K_train_HR"
crop_size = 512
scaled_size = 256
num_selected = 10

# Get all image files
image_files = sorted(glob.glob(os.path.join(dataset_path, "*.png")))
print(f"Found {len(image_files)} images")

# Function for fast BPP computation with batching (much faster)
def compute_bpp_batch(img_paths, crop_size=512, scaled_size=256, batch_size=4):
    """Compute BPP for multiple images in batches (much faster than single image processing)"""
    results = []
    preprocess = transforms.Compose([transforms.ToTensor()])
    num_batches = (len(img_paths) + batch_size - 1) // batch_size
    
    for batch_start in tqdm(range(0, len(img_paths), batch_size), desc="Computing BPP", unit="batch", total=num_batches):
        batch_paths = img_paths[batch_start:batch_start + batch_size]
        batch_lr_tensors = []
        batch_info = []
        
        # Load and preprocess batch
        for img_path in batch_paths:
            try:
                img = Image.open(img_path).convert('RGB')
                width, height = img.size
                
                if width >= crop_size and height >= crop_size:
                    # Center crop
                    left = (width - crop_size) // 2
                    top = (height - crop_size) // 2
                    cropped_img = img.crop((left, top, left + crop_size, top + crop_size))
                    
                    # Create LR version
                    low_res_img = cropped_img.resize((scaled_size, scaled_size), Image.BICUBIC)
                    
                    # Convert to tensor
                    lr_tensor = preprocess(low_res_img).unsqueeze(0)
                    batch_lr_tensors.append(lr_tensor)
                    batch_info.append({
                        'filename': os.path.basename(img_path),
                        'path': img_path
                    })
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                continue
        
        if not batch_lr_tensors:
            continue
        
        # Stack into batch tensor
        batch_tensor = torch.cat(batch_lr_tensors, dim=0).to(device)
        
        # Apply super-resolution to batch
        with torch.no_grad():
            sr_output = model_sr(batch_tensor)
            sr_tensor = sr_output.reconstruction
        
        # Apply compression to entire batch (much faster!)
        with torch.no_grad():
            compression_output = model_compression(sr_tensor)
            
            # Compute BPP for each image in batch
            num_pixels = sr_tensor.shape[2] * sr_tensor.shape[3]
            for i in range(sr_tensor.shape[0]):
                # Extract likelihoods for this image from batch
                y_likelihood = compression_output['likelihoods']['y'][i]
                z_likelihood = compression_output['likelihoods']['z'][i]
                bpp = (torch.log(y_likelihood).sum() + torch.log(z_likelihood).sum()) / (-math.log(2) * num_pixels)
                
                results.append({
                    'filename': batch_info[i]['filename'],
                    'path': batch_info[i]['path'],
                    'bpp': bpp.item()
                })
    
    return results

# Function for processing selected images with full SR and tensor storage
def process_selected_image(img_path, crop_size=512, scaled_size=256):
    """Process selected image with full SR and tensor storage"""
    try:
        # Load and crop
        img = Image.open(img_path).convert('RGB')
        width, height = img.size
        
        if width >= crop_size and height >= crop_size:
            # Center crop
            left = (width - crop_size) // 2
            top = (height - crop_size) // 2
            cropped_img = img.crop((left, top, left + crop_size, top + crop_size))
            
            # Create LR version
            low_res_img = cropped_img.resize((scaled_size, scaled_size), Image.BICUBIC)
            
            # Convert to tensors
            preprocess = transforms.Compose([transforms.ToTensor()])
            hr_tensor = preprocess(cropped_img).unsqueeze(0).to(device)
            lr_tensor = preprocess(low_res_img).unsqueeze(0).to(device)
            
            # Apply super-resolution
            with torch.no_grad():
                sr_output = model_sr(lr_tensor)
                sr_tensor = sr_output.reconstruction
            
            # Compute BPP again for verification
            with torch.no_grad():
                compression_output = model_compression(sr_tensor)
                num_pixels = sr_tensor.shape[2] * sr_tensor.shape[3]
                bpp = bpp_loss(compression_output, num_pixels).item()
            
            filename = os.path.basename(img_path)
            return {
                'filename': filename,
                'path': img_path,
                'bpp': bpp,
                'hr_tensor': hr_tensor.cpu(),
                'lr_tensor': lr_tensor.cpu(),
                'sr_tensor': sr_tensor.cpu()
            }
        else:
            return None
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

print("Functions ready")

Found 800 images
Functions ready


In [12]:
# Step 1: Compute BPP for all images using batching (much faster)
print("Step 1: Computing BPP for all images (fast pass with batching)...")
batch_size = 8  # Increased batch size for better performance (adjust based on available memory)

# Process all images with batching (progress bar is inside the function)
bpp_results = compute_bpp_batch(image_files, crop_size, scaled_size, batch_size)

# Sort by BPP
print("\nSorting by BPP...")
bpp_results_sorted = sorted(bpp_results, key=lambda x: x['bpp'])
print(f"Successfully processed {len(bpp_results_sorted)} images")
print(f"BPP range: {bpp_results_sorted[0]['bpp']:.4f} - {bpp_results_sorted[-1]['bpp']:.4f}")

Step 1: Computing BPP for all images (fast pass with batching)...


Computing BPP:   3%|▎         | 3/100 [04:38<2:30:08, 92.87s/batch]


KeyboardInterrupt: 

In [ ]:
# Step 2: Select 10 diverse images based on BPP distribution
print("Step 2: Selecting 10 diverse images...")
selected_paths = []
if len(bpp_results_sorted) >= num_selected:
    group_size = len(bpp_results_sorted) // num_selected
    for i in range(num_selected):
        idx = i * group_size + group_size // 2
        if idx < len(bpp_results_sorted):
            selected_paths.append(bpp_results_sorted[idx])
    
    print(f"Selected {len(selected_paths)} images:")
    for img in selected_paths:
        print(f"  {img['filename']}: BPP = {img['bpp']:.4f}")
else:
    print(f"Not enough images to select {num_selected}")

In [ ]:
# Step 3: Apply SR to selected images and save results
print("\nStep 3: Applying super-resolution to selected images...")
selected_images_full = []

for img_info in tqdm(selected_paths, desc="Processing selected images", unit="image"):
    result = process_selected_image(img_info['path'], crop_size, scaled_size)
    if result is not None:
        selected_images_full.append(result)

# Save results
import pickle
selection_results = {
    'all_bpp_results': bpp_results_sorted,
    'selected_images': selected_images_full
}

with open('div2k_bpp_sorted_selection.pkl', 'wb') as f:
    pickle.dump(selection_results, f)

print(f"\nResults saved to 'div2k_bpp_sorted_selection.pkl'")
print(f"Total images processed: {len(bpp_results_sorted)}")
print(f"Selected images with full SR: {len(selected_images_full)}")